# DWS 层 — 日粒度聚合

`fact_trips` → 按日 × 区域 × 来源 聚合，产出 `dws/trips_daily`。

两个核心优化：
1. **加盐两阶段聚合**：热点区域（如 161/237）数据倾斜，加随机盐先局部聚合，再合并
2. **滑动窗口处理跨日迟到数据**：重算时往前多取一天，覆盖迟到记录

## 1. SparkSession

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, year, month, dayofmonth

spark = (
    SparkSession.builder
    .appName("TLC-DWS-Build")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

BASE     = os.path.expanduser("~/tlc-spark-pipeline")
DWD_FACT = f"{BASE}/data/warehouse/dwd/fact_trips"
DWS_OUT  = f"{BASE}/data/warehouse/dws/trips_daily"

TARGET_DATE = os.environ.get("TARGET_DATE", "2024-03-15")
yr  = int(TARGET_DATE.split("-")[0])
mo  = int(TARGET_DATE.split("-")[1])
day = int(TARGET_DATE.split("-")[2])
print(f"TARGET_DATE={TARGET_DATE}")

print(f"Spark {spark.version} ready")

26/10/19 14:57:23 WARN Utils: Your hostname, jelly-macbook.local resolves to a loopback address: 127.0.0.1; using 192.168.1.229 instead (on interface en0)
26/10/19 14:57:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/10/19 14:57:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


TARGET_DATE=2024-03-15
Spark 3.5.1 ready


## 2. 读取 fact_trips + 确认热点区域倾斜

In [2]:
fact = (
    spark.read.parquet(DWD_FACT)
    .filter((col("pickup_year") == yr) & (col("pickup_month") == mo) & (col("pickup_day") == day))
)
print(f"fact_trips: {fact.count():,} rows")

print("\n=== 热点区域（Top 10 上车量）===")
fact.groupBy("PULocationID") \
    .agg(F.count("*").alias("trip_count")) \
    .orderBy(F.desc("trip_count")) \
    .limit(10) \
    .show()

fact_trips: 862,909 rows

=== 热点区域（Top 10 上车量）===


+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         132|     16910|
|         138|     15570|
|          79|     13217|
|         161|     12637|
|         230|     12302|
|          68|     11168|
|         237|     11082|
|         170|     10897|
|         164|     10852|
|         236|     10794|
+------------+----------+



## 3. 加盐两阶段聚合（解决热点区域数据倾斜）

热点区域（如 161 号区域拥有 40 万+ 条记录）会导致 groupBy 时某个 partition 数据量远超其他，形成长尾。

**解决方案：**
1. 第一阶段：给 PULocationID 加随机盐（0~7），先做局部聚合，将一个大 key 拆成 8 个小 key
2. 第二阶段：去掉盐，再做一次合并聚合，得到真实的每日区域汇总

In [3]:
SALT_BUCKETS = 8

# 增加日期列
fact_with_date = (
    fact
    .withColumn("pickup_date", to_date("pickup_datetime"))
)

# 第一阶段：加盐局部聚合
phase1 = (
    fact_with_date
    .withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int"))
    .groupBy("pickup_date", "PULocationID", "source_type", "salt")
    .agg(
        F.count("*").alias("trip_count"),
        F.sum("total_amount").alias("gmv"),
        F.sum("fare_amount").alias("fare_sum"),
        F.sum("tip_amount").alias("tip_sum"),
        F.sum("surcharge_total").alias("surcharge_sum"),
        F.sum("trip_distance").alias("distance_sum"),
        F.sum("trip_duration_min").alias("duration_sum")
    )
)

# 第二阶段：去盐合并
dws = (
    phase1
    .groupBy("pickup_date", "PULocationID", "source_type")
    .agg(
        F.sum("trip_count").alias("trip_count"),
        F.round(F.sum("gmv"), 2).alias("gmv"),
        F.round(F.sum("fare_sum"), 2).alias("fare_sum"),
        F.round(F.sum("tip_sum"), 2).alias("tip_sum"),
        F.round(F.sum("surcharge_sum"), 2).alias("surcharge_sum"),
        F.round(F.sum("distance_sum"), 2).alias("distance_sum"),
        F.round(F.sum("duration_sum"), 2).alias("duration_sum")
    )
    .withColumn("pickup_year",  year("pickup_date"))
    .withColumn("pickup_month", month("pickup_date"))
    .withColumn("pickup_day",   dayofmonth("pickup_date"))
)

dws_count = dws.count()
print(f"DWS rows: {dws_count:,}")
dws.show(5)

DWS rows: 559


+-----------+------------+-----------+----------+---------+---------+--------+-------------+------------+------------+-----------+------------+----------+
|pickup_date|PULocationID|source_type|trip_count|      gmv| fare_sum| tip_sum|surcharge_sum|distance_sum|duration_sum|pickup_year|pickup_month|pickup_day|
+-----------+------------+-----------+----------+---------+---------+--------+-------------+------------+------------+-----------+------------+----------+
| 2024-03-15|         159|     fhv_hv|      3595| 76899.26| 73180.68|  833.83|        736.0|    13709.12|    67609.09|       2024|           3|        15|
| 2024-03-15|         186|     fhv_hv|      4934|216828.96|176598.22| 10951.7|     12834.75|     31334.7|   123549.73|       2024|           3|        15|
| 2024-03-15|          17|     fhv_hv|      5966|137169.07|128758.03| 4626.98|      2014.25|    21626.53|   116239.06|       2024|           3|        15|
| 2024-03-15|         131|     fhv_hv|      1288| 30820.97| 29211.63| 

## 4. 滑动窗口处理跨日迟到数据

按日增量跑任务时，某些行程的数据可能在次日才到达（如深夜行程跨零点上报）。

**处理方式：** 重算 T 日时，实际查询范围是 `[T-1 23:00, T+1 01:00]`，覆盖跨日边界，然后只输出 pickup_date = T 的记录，通过动态分区覆盖保证幂等。

In [4]:
# 演示：以 2024-01-15 为目标日期，展示滑动窗口逻辑
TARGET_DATE = "2024-01-15"

dws_single_day = (
    fact_with_date
    # 滑动窗口：取前后各 1 小时，覆盖跨日迟到
    .filter(col("pickup_datetime") >= F.expr(f"timestamp('{TARGET_DATE}') - interval 1 hours"))
    .filter(col("pickup_datetime") <  F.expr(f"timestamp('{TARGET_DATE}') + interval 25 hours"))
    # 只保留目标日期的记录输出
    .filter(col("pickup_date") == TARGET_DATE)
    .withColumn("salt", (F.rand() * SALT_BUCKETS).cast("int"))
    .groupBy("pickup_date", "PULocationID", "source_type", "salt")
    .agg(F.count("*").alias("trip_count"), F.sum("total_amount").alias("gmv"),
         F.sum("fare_amount").alias("fare_sum"), F.sum("tip_amount").alias("tip_sum"),
         F.sum("surcharge_total").alias("surcharge_sum"),
         F.sum("trip_distance").alias("distance_sum"), F.sum("trip_duration_min").alias("duration_sum"))
    .groupBy("pickup_date", "PULocationID", "source_type")
    .agg(F.sum("trip_count").alias("trip_count"), F.round(F.sum("gmv"), 2).alias("gmv"),
         F.round(F.sum("fare_sum"), 2).alias("fare_sum"), F.round(F.sum("tip_sum"), 2).alias("tip_sum"),
         F.round(F.sum("surcharge_sum"), 2).alias("surcharge_sum"),
         F.round(F.sum("distance_sum"), 2).alias("distance_sum"),
         F.round(F.sum("duration_sum"), 2).alias("duration_sum"))
    .withColumn("pickup_year",  year("pickup_date"))
    .withColumn("pickup_month", month("pickup_date"))
    .withColumn("pickup_day",   dayofmonth("pickup_date"))
)

print(f"{TARGET_DATE} zones: {dws_single_day.count()}")
dws_single_day.orderBy(F.desc("trip_count")).show(5)

2024-01-15 zones: 0


+-----------+------------+-----------+----------+---+--------+-------+-------------+------------+------------+-----------+------------+----------+
|pickup_date|PULocationID|source_type|trip_count|gmv|fare_sum|tip_sum|surcharge_sum|distance_sum|duration_sum|pickup_year|pickup_month|pickup_day|
+-----------+------------+-----------+----------+---+--------+-------+-------------+------------+------------+-----------+------------+----------+
+-----------+------------+-----------+----------+---+--------+-------+-------------+------------+------------+-----------+------------+----------+



## 5. 写出 DWS（动态分区覆盖）

In [5]:
%%time
(
    dws
    .write
    .partitionBy("pickup_year", "pickup_month", "pickup_day")
    .mode("overwrite")
    .parquet(DWS_OUT)
)
print(f"DWS written to: {DWS_OUT}")

# 验证
dws_check = spark.read.parquet(DWS_OUT)
print(f"DWS total rows: {dws_check.count():,}")
dws_check.groupBy("pickup_year", "pickup_month", "source_type") \
    .agg(F.sum("trip_count").alias("trips"), F.round(F.sum("gmv"), 0).alias("gmv")) \
    .orderBy("pickup_year", "pickup_month", "source_type").show()

spark.stop()
print("DWS build complete.")

26/10/19 14:57:40 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


DWS written to: /Users/lujiayi/tlc-spark-pipeline/data/warehouse/dws/trips_daily


DWS total rows: 559


+-----------+------------+-----------+------+-----------+
|pickup_year|pickup_month|source_type| trips|        gmv|
+-----------+------------+-----------+------+-----------+
|       2024|           3|     fhv_hv|739978|2.2096175E7|
|       2024|           3|      green|  1868|    42972.0|
|       2024|           3|     yellow|121063|  3401357.0|
+-----------+------------+-----------+------+-----------+



DWS build complete.
CPU times: user 19.4 ms, sys: 11.4 ms, total: 30.8 ms
Wall time: 2.79 s
